<a href="https://colab.research.google.com/github/GuiCastro7/Grupo-3---ECAA08/blob/main/06_Quantificadores_e_Predicados_em_Redes_de_Sensores_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 06 - Notebook: Lógica de Predicados e Quantificadores em Redes de Sensores
Neste notebook implementamos a avaliação de predicados unários e binários e os quantificadores $\forall$ (FORALL) e $\exists$ (EXISTS) sobre coleções de sensores distribuídos na **Linha de Envase e Tampamento de Garrafas**.

In [ ]:
from dataclasses import dataclass
from typing import List, Callable, Any

def formatar_tabela(dados):
    """Formata lista de dicionarios em tabela ASCII pura."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)

@dataclass
class SensorProcesso:
    tag: str
    setor: str
    tipo: str
    valor: float
    unidade: str
    limite: float
    falha_comunicacao: bool = False

def FORALL(dominio: List[Any], predicado: Callable[[Any], bool]) -> bool:
    return all(predicado(x) for x in dominio)

def EXISTS(dominio: List[Any], predicado: Callable[[Any], bool]) -> bool:
    return any(predicado(x) for x in dominio)

In [ ]:
# Rede de sensores da linha de envase (com injeção de sobrepressão em SP1 para validação)
rede_sensores = [
    SensorProcesso('SP1', 'Alimentacao TS1', 'PRESSAO', 3.8, 'Barg', 3.5),
    SensorProcesso('SQ1', 'Alimentacao TS1', 'VAZAO', 42.0, 'L/min', 45.0),
    SensorProcesso('SP2', 'Acumulador AS1', 'PRESSAO', 4.1, 'Barg', 4.5),
    SensorProcesso('SQ2', 'Linha de Envase', 'VAZAO', 6.5, 'L/min', 8.0),
    SensorProcesso('SL1', 'Estacao Enchimento', 'NIVEL', 96.0, '%', 95.0),
    SensorProcesso('SFC1', 'Estacao Capping', 'FIM_CURSO', 1.0, 'Digital', 1.0)
]

# Filtro de transmissores de pressão
sensores_pressao = [s for s in rede_sensores if s.tipo == 'PRESSAO']

# Avaliação dos Quantificadores
existe_sobrepressao = EXISTS(sensores_pressao, lambda s: s.valor >= s.limite)
todos_comunicando = FORALL(rede_sensores, lambda s: not s.falha_comunicacao)

print(f"1. Existe Sobrepressao na Linha (EXISTS): {existe_sobrepressao}")
print(f"2. Todos Sensores Comunicando (FORALL): {todos_comunicando}")

# Geração da Tabela ASCII de Diagnóstico
tabela = [
    {
        "Tag": s.tag,
        "Setor": s.setor,
        "Tipo": s.tipo,
        "Valor": f"{s.valor} {s.unidade}",
        "Limite": f"{s.limite} {s.unidade}",
        "Falha": (s.valor >= s.limite) if s.tipo == 'PRESSAO' else False
    }
    for s in rede_sensores
]

print("\n" + formatar_tabela(tabela))
assert existe_sobrepressao is True